## Cell 0A — Google Drive Mount
Run this first, every session. Authenticates and mounts your Drive.
Safe to re-run — uses `force_remount=False`.

In [ ]:
# ── GOOGLE DRIVE MOUNT ───────────────────────────────────────────────────────
# Run this cell first, every Colab session.
# Safe to re-run — force_remount=False skips the auth dialog if already mounted.
# If you see "Mountpoint must not already contain files", the cleanup block
# below clears the stale mount automatically.

import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
try:
    from google.colab import drive as _drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
        print('Drive mounted at /content/drive')
    except Exception as _e:
        if 'already contain files' in str(_e) or 'symlink' in str(_e):
            print('Stale mount detected — clearing and remounting...')
            subprocess.run(['umount', '/content/drive'], capture_output=True)
            subprocess.run(['rm', '-rf', '/content/drive'], capture_output=True)
            drive.mount('/content/drive', force_remount=False)
            print('Drive remounted successfully.')
        else:
            raise
else:
    print('Not running in Colab -- Drive mount skipped.')
    print('Set paths in Cell 0C to point to your local project folder.')


## Cell 0B — Install Dependencies
Installs packages not pre-loaded in Colab. Takes ~60 s on first run; instant on subsequent runs (pip skips already-installed packages).

In [ ]:
# ── INSTALL PIPELINE DEPENDENCIES ───────────────────────────────────────────
# Pre-installed in Colab: numpy, scipy, pandas, matplotlib, opencv-python
# Needs installing: papermill, pyarrow, tqdm
# Optional (uncomment when needed):
#   ultralytics   -- YOLOv8 neural segmentation (NB-02 neural mode)
#   torch         -- PyTorch for NB-04 LSTM training
#   xgboost       -- regime classifier (NB-05 neural mode)
#   motmetrics    -- CLEAR-MOT tracking metrics (NB-03 gate)

import subprocess, sys

PACKAGES = [
    'papermill',
    'pyarrow',
    'tqdm',
    'pycocotools',
]

# Optional -- uncomment as needed:
# PACKAGES += ['ultralytics']
# PACKAGES += ['torch']
# PACKAGES += ['xgboost', 'scikit-learn']
# PACKAGES += ['motmetrics']

print('Installing packages...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + PACKAGES,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('pip error:', result.stderr[-500:])
else:
    print('Done:  ' + '  '.join(PACKAGES))


## Cell 0C — Path Configuration
Sets `DRIVE_ROOT` and `SCRATCH_ROOT` for this session.
**Edit `DRIVE_PROJECT_PATH`** if your BubbleFlow folder is in a different location.

| Variable | Lives on | Survives reset? |
|---|---|---|
| `DRIVE_ROOT` | Google Drive | ✅ Yes |
| `SCRATCH_ROOT` | Colab VM `/content/` | ❌ No — fast temp storage |

In [ ]:
# ── PATH CONFIGURATION ───────────────────────────────────────────────────────
# Edit DRIVE_PROJECT_PATH to match where you created the BubbleFlow folder
# on your Google Drive.
#
# Default (Drive root):  'BubbleFlow'
# Subfolder example:     'Research/BubbleColumn/BubbleFlow'

from pathlib import Path
import sys

IN_COLAB = 'google.colab' in sys.modules
try:
    from google.colab import drive as _d
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── EDIT THIS LINE ────────────────────────────────────────────────────────────
DRIVE_PROJECT_PATH = 'SIGNAL_NN_2026/HIDRO_2026'
# ─────────────────────────────────────────────────────────────────────────────

if IN_COLAB:
    DRIVE_ROOT   = Path('/content/drive/MyDrive') / DRIVE_PROJECT_PATH
    SCRATCH_ROOT = Path('/content/scratch')
else:
    # Local Jupyter: set this to your local BubbleFlow folder
    DRIVE_ROOT   = Path.home() / 'BubbleFlow'
    SCRATCH_ROOT = Path('/tmp/bubbleflow_scratch')

# Create scratch directories (lost on reset -- recreated each session)
for sub in ['preproc', 'segmentation', 'tracking', 'temporal', 'flow']:
    (SCRATCH_ROOT / sub).mkdir(parents=True, exist_ok=True)

# ── Print resolved paths so you can verify before running Cell 1 ─────────────
print('Session paths')
print('  DRIVE_ROOT   :', DRIVE_ROOT)
print('  SCRATCH_ROOT :', SCRATCH_ROOT)
print('  Drive exists :', DRIVE_ROOT.exists())
print()
if not DRIVE_ROOT.exists():
    print('WARNING: DRIVE_ROOT does not exist.')
    print('  Create the folder on Drive, or change DRIVE_PROJECT_PATH above.')
else:
    # List top-level contents
    items = list(DRIVE_ROOT.iterdir())
    print('Contents of DRIVE_ROOT:')
    for item in sorted(items)[:20]:
        tag = '[dir] ' if item.is_dir() else '[file]'
        print('  ' + tag + ' ' + item.name)


# NB-00 · Physical Calibration & ROI Definition
### Bubble Flow Analysis System — Stage 0

**Roadmap gate:** No downstream notebook may run without a validated, version-stamped `config.json`.  
**Acceptance criterion:** Calibration error < 2 % against a known physical reference dimension.

---
### Calibration target

A black-and-white checkerboard is mounted inside the test column.  
**Each square has a 40 mm (4 cm) side.**  The board is 11 × 10 squares = 440 × 400 mm.

`cv2.findChessboardCorners()` detects all 90 inner corner intersections automatically  
with sub-pixel refinement, providing up to **89 independent spacing measurements** per frame.  
Relative calibration uncertainty is typically 1–1.5 %, well inside the 2 % gate.

---
**Inputs required**
- A reference frame (JPEG / PNG) containing the checkerboard inside the column
- Video metadata: FPS, resolution (width × height in pixels)
- Physical column inner diameter (mm)

**Output**
- `config.json` — mandatory schema v1.0 (Appendix A of the roadmap)


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import math
import warnings
from datetime import datetime, timezone
from pathlib import Path

# ── Scientific stack ──────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.widgets import RectangleSelector
from scipy import stats as sp_stats

matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore', category=UserWarning)

print('NB-00  ·  Calibration  ·  imports OK')

---
## Cell 1 — User Configuration
Edit the variables in this cell before running anything else.

### Required run order
**Run every cell in sequence from top to bottom.** Each cell creates variables the next one needs:

| Cell | Creates |
|---|---|
| 0A | Drive mount |
| 0B | Installed packages |
| 0C | `DRIVE_ROOT`, `SCRATCH_ROOT` |
| 1  | All config variables |
| 2  | Function definitions |
| 3  | `frame_bgr`, **`scale_result`** |
| 4  | `calib` (mm_per_pixel + uncertainty) |
| 5  | `validated_fps` |
| **6** | **`roi_coords`, `reference_lines`** — *needs `scale_result` from Cell 3* |
| 7  | Overlay plot + PNG saved to Drive |
| 8  | `gate_pass` |
| 9  | `config.json` written to Drive |
| 10 | Round-trip validation |
| 11 | Summary report |

In [ ]:
# ── EDIT THESE PATHS AND VALUES ─────────────────────────────────────────────
#
# REFERENCE_FRAME_PATH — use the FULL absolute path.
# Examples:
#   Local Jupyter : "/home/user/BubbleFlow/campaigns/setup_A/reference_frame.jpg"
#   Google Colab  : "/content/drive/MyDrive/BubbleFlow/campaigns/setup_A/reference_frame.jpg"
#   Same folder   : "reference_frame.jpg" (only if notebook is in the same folder)

REFERENCE_FRAME_PATH: str = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A/reference_frame.jpg"   # <- set YOUR path

# ── CHECKERBOARD SETTINGS ────────────────────────────────────────────────────
# Each square side = 40 mm  (confirmed from lab photo)
# Board: 11 x 10 squares visible  ->  10 x 9 inner corners
SQUARE_SIZE_MM: float     = 40.0     # physical side of one square (mm)
CHECKERBOARD_GRID: tuple  = (10, 9)  # inner corners (cols, rows)
N_SQUARES_REFERENCE: int  = 11
KNOWN_PHYSICAL_LENGTH_MM: float = COLUMN_WIDTH_CALIB_MM   # = 500.0  manual mode uses column width

# Detection mode:
#   "auto"         try checkerboard first, fall back to canny_hough
#   "checkerboard" force checkerboard (error if board not detected)
#   "canny_hough"  legacy Hough line detection
CALIBRATION_MODE: str = "auto"

# ── COLUMN AND VIDEO METADATA ─────────────────────────────────────────────────
COLUMN_DIAMETER_MM: float = 500.0     # <- measure with calliper
COLUMN_WIDTH_CALIB_MM = 500.0
LEFT_WALL_PX          : int   = 0      # read from Cell 3a diagnostic
RIGHT_WALL_PX         : int   = 2160      # read from Cell 3a diagnostic
VIDEO_FPS: float          = 44.638   # <- camera frame rate
VIDEO_WIDTH_PX: int       = 2160     # <- frame width  in pixels
VIDEO_HEIGHT_PX: int      = 3840     # <- frame height in pixels
REFERENCE_VIDEO_PATH: str = ""       # <- optional video for FPS validation
OUTPUT_DIR: str           = "campaigns/setup_A"  # <- config.json goes here
REFERENCE_LINES_PX: list  = []       # e.g. [300, 500, 700]
SCHEMA_VERSION: str       = "1.0"

# ── GATE THRESHOLD ───────────────────────────────────────────────────────────
# Maximum acceptable relative calibration error (%).
# Downstream notebooks are blocked if this threshold is exceeded.
MAX_RELATIVE_CALIBRATION_ERROR_PCT: float = 2.0

# ── ROI INPUT TIMEOUT ────────────────────────────────────────────────────────
# Seconds to wait for the user to type ROI coordinates before falling back
# to the full frame.  120 seconds = 2 minutes.
# ── ROI INPUT TIMEOUT ────────────────────────────────────────────────────────
ROI_INPUT_TIMEOUT_S: int  = 300   # 5 minutes

# ── ROI DIRECT OVERRIDE (fastest option — avoids the prompt entirely) ────────
# Set these before running Cell 6.  If non-empty they are used immediately
# and the interactive prompt is SKIPPED completely.
#
# Portrait 4K video (2160 x 3840):  column body coordinates
ROI_COORDS_OVERRIDE: list  = [0, 450, 2160, 3400]  # [x0,y0,x1,y1]
REF_LINES_OVERRIDE:  list  = [1187, 1925, 2662]       # Y px at 25/50/75%

# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_PATH : str = str(DRIVE_ROOT / 'campaigns/setup_A')
Path(OUTPUT_PATH).mkdir(parents=True, exist_ok=True)
print(f"Output directory  : {OUTPUT_PATH}")
print(f"Reference frame   : {REFERENCE_FRAME_PATH}")
print(f"  -> exists        : {Path(REFERENCE_FRAME_PATH).exists()}")
print(f"Calibration mode  : {CALIBRATION_MODE}")
print(f"Checkerboard grid : {CHECKERBOARD_GRID}  ({CHECKERBOARD_GRID[0]+1}x{CHECKERBOARD_GRID[1]+1} squares)")
print(f"Square size       : {SQUARE_SIZE_MM} mm")
print(f"Reference length  : {KNOWN_PHYSICAL_LENGTH_MM} mm  ({N_SQUARES_REFERENCE} x {SQUARE_SIZE_MM} mm)")
print(f"Column diameter   : {COLUMN_DIAMETER_MM} mm")
print(f"Video FPS         : {VIDEO_FPS}")
print(f"Resolution        : {VIDEO_WIDTH_PX} x {VIDEO_HEIGHT_PX} px")


In [ ]:
import shutil
from pathlib import Path
old = Path('/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026'
           '/campaigns/setup_A/campaigns/outputs/preproc')
if old.exists():
    shutil.rmtree(old)
    print('Old preproc deleted.')

---
## Cell 2 — Core Helper Functions
All functions named exactly as specified in the roadmap (Section 7, NB-00).

Functions defined here:
- `detect_scale_object()` — two-pass checkerboard + Hough fallback
- `compute_mm_per_pixel()` — scale factor computation
- `quantify_calibration_uncertainty()` — Student-t CI on N measurements
- `define_roi_interactive()` — Colab-safe ROI selection with timeout
- `plot_calibration_overlay()` — full annotated overlay image
- `save_config()` / `load_config()` — config.json read/write

In [ ]:
# ── detect_scale_object() ─────────────────────────────────────────────────────

def detect_scale_object(
    frame: np.ndarray,
    method: str = 'auto',
    grid: tuple = (10, 9),
    min_length_px: int = 50,
    debug: bool = True,
) -> dict:
    """
    Detect the physical scale reference and return its measured pixel length.

    Parameters
    ----------
    frame         : BGR image as returned by cv2.imread
    method        : 'auto'        — checkerboard first, Hough fallback
                    'checkerboard'— OpenCV findChessboardCorners (primary)
                    'canny_hough' — Canny + Probabilistic Hough Lines (fallback)
                    'manual'      — returns pixel_length = 0 (legacy, kept for
                                    compatibility; Cell 3b has been retired)
    grid          : (cols, rows) of inner corners for checkerboard method
    min_length_px : minimum Hough line length to keep (canny_hough only)
    debug         : if True, annotate a debug image with detections

    Returns
    -------
    dict with keys
        'pixel_length'       : best-estimate pixel length of the reference object
        'pixel_lengths_all'  : list of all individual measurements (for uncertainty)
        'method'             : method actually used
        'n_corners'          : number of checkerboard corners found (0 if Hough)
        'lines'              : raw Hough segments (empty if checkerboard)
        'board_corners'      : 4-corner board boundary [[x,y],...] or None
        'debug_image'        : annotated BGR image (or None)
    """
    result = {
        'pixel_length':      0.0,
        'pixel_lengths_all': [],
        'method':            method,
        'n_corners':         0,
        'lines':             [],
        'board_corners':     None,
        'debug_image':       None,
    }

    if method == 'manual':
        print("detect_scale_object: 'manual' mode is retired — "
              "checkerboard or Hough used automatically.")
        return result

    debug_img = frame.copy()

    # ── CHECKERBOARD PATH ─────────────────────────────────────────────────────
    def _try_checkerboard(frm, g):
        # Two-pass detection.
        # Pass 1: full image  (works when board fills most of the frame).
        # Pass 2: statistical scan  (needed when reference photo has large
        #   non-board areas: glass frame, reflections, lab background).
        #   Scans sub-regions with checkerboard statistics (mean 85-150,
        #   std > 55) then translates corners back to full-image coords.
        gray_frm = cv2.cvtColor(frm, cv2.COLOR_BGR2GRAY)
        fh, fw = gray_frm.shape[:2]
        det_flags = (
            cv2.CALIB_CB_ADAPTIVE_THRESH +
            cv2.CALIB_CB_NORMALIZE_IMAGE +
            cv2.CALIB_CB_FAST_CHECK
        )
        crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

        # Pass 1: full image
        ret, corners = cv2.findChessboardCorners(gray_frm, g, det_flags)
        if ret and corners is not None:
            corners = cv2.cornerSubPix(gray_frm, corners, (11, 11), (-1, -1), crit)
            print("  Checkerboard detected on full image.")
            return ret, corners, 0, 0

        # Pass 2: statistical scan
        print("  Full-image pass failed -- scanning sub-regions for the board...")
        step = 50
        for y0 in range(0, fh - 200, step):
            for x0 in range(0, fw - 200, step):
                for frac_h in [0.65, 0.75, 0.85]:
                    for frac_w in [0.40, 0.50, 0.60, 0.70]:
                        y1 = min(fh, y0 + int(fh * frac_h))
                        x1 = min(fw, x0 + int(fw * frac_w))
                        region = gray_frm[y0:y1, x0:x1]
                        m = float(region.mean())
                        s = float(region.std())
                        if 85 < m < 150 and s > 55:
                            ret2, c2 = cv2.findChessboardCorners(region, g, det_flags)
                            if ret2 and c2 is not None:
                                c2 = cv2.cornerSubPix(
                                    region, c2, (11, 11), (-1, -1), crit
                                )
                                print("  Found in sub-region:"
                                      " offset x=%d y=%d  size=%dx%d"
                                      % (x0, y0, x1 - x0, y1 - y0))
                                return ret2, c2, x0, y0
        print("  Checkerboard not found in any sub-region -- Hough fallback will run.")
        return None, None, 0, 0

    use_checkerboard = method in ('auto', 'checkerboard')

    if use_checkerboard:
        ret, corners, off_x, off_y = _try_checkerboard(frame, grid)

        if ret and corners is not None:
            # Translate corners from sub-region back to full-image coordinates
            if off_x > 0 or off_y > 0:
                import numpy as _np
                corners = corners + _np.array([[[off_x, off_y]]], dtype=_np.float32)
            pts = corners.reshape(grid[1], grid[0], 2)  # (rows, cols, 2)

            # All horizontal inter-corner distances (same row, adjacent cols)
            h_spacings = [
                float(np.linalg.norm(pts[r, c+1] - pts[r, c]))
                for r in range(grid[1])
                for c in range(grid[0] - 1)
            ]
            # All vertical inter-corner distances (same col, adjacent rows)
            v_spacings = [
                float(np.linalg.norm(pts[r+1, c] - pts[r, c]))
                for r in range(grid[1] - 1)
                for c in range(grid[0])
            ]
            all_spacings = h_spacings + v_spacings  # 89 values for a 10×9 grid

            # Best-estimate pixel span of ONE square (mean spacing)
            mean_sq_px = float(np.mean(all_spacings))

            # Board boundary corners for ROI auto-detection
            tl = pts[0,  0].tolist()
            tr = pts[0, -1].tolist()
            bl = pts[-1, 0].tolist()
            br = pts[-1,-1].tolist()

            result['method']             = 'checkerboard'
            result['n_corners']          = len(corners)
            result['pixel_length']       = mean_sq_px   # px per one square
            result['pixel_lengths_all']  = all_spacings  # passed to uncertainty fn
            result['board_corners']      = [tl, tr, br, bl]

            if debug:
                cv2.drawChessboardCorners(debug_img, grid, corners, ret)
                # Draw board boundary
                bnd = np.array([tl, tr, br, bl], dtype=np.int32)
                cv2.polylines(debug_img, [bnd], True, (0, 255, 0), 2)
                # Annotate one square with its size
                cx = int(pts[grid[1]//2, grid[0]//2, 0])
                cy = int(pts[grid[1]//2, grid[0]//2, 1])
                cv2.putText(
                    debug_img,
                    f"sq={mean_sq_px:.1f}px  n={len(all_spacings)}",
                    (cx - 80, cy - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2,
                )
            result['debug_image'] = debug_img
            print(f"Checkerboard detected: {len(corners)} corners  "
                  f"mean_sq={mean_sq_px:.2f} px  n_spacings={len(all_spacings)}")
            return result

        else:
            if method == 'checkerboard':
                print("[ERROR] Checkerboard not detected. "
                      "Check CHECKERBOARD_GRID and image quality.")
                result['debug_image'] = debug_img
                return result
            else:  # 'auto' — fall through to Hough
                print("[INFO] Checkerboard not found — falling back to Hough.")

    # ── CANNY + HOUGH PATH (fallback) ─────────────────────────────────────────
    gray    = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    otsu_thresh, _ = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    canny_lo = max(0.33 * otsu_thresh, 30)
    canny_hi = min(1.33 * otsu_thresh, 255)
    edges    = cv2.Canny(blurred, int(canny_lo), int(canny_hi))
    hough_lines = cv2.HoughLinesP(
        edges, rho=1, theta=np.pi/180,
        threshold=50, minLineLength=min_length_px, maxLineGap=15,
    )
    if hough_lines is None:
        print("detect_scale_object: no Hough lines found.")
        result['debug_image'] = debug_img
        return result

    segments = [tuple(l[0]) for l in hough_lines]
    result['lines'] = segments
    lengths  = [math.hypot(x2-x1, y2-y1) for x1,y1,x2,y2 in segments]
    best_idx = int(np.argmax(lengths))
    result['pixel_length']      = lengths[best_idx]
    result['pixel_lengths_all'] = [lengths[best_idx]]  # single measurement
    result['method']            = 'canny_hough'

    if debug:
        for k, (x1,y1,x2,y2) in enumerate(segments):
            color = (0, 255, 0) if k == best_idx else (100, 100, 100)
            cv2.line(debug_img, (x1,y1), (x2,y2), color, 2)
        x1,y1,x2,y2 = segments[best_idx]
        cv2.putText(debug_img, f"{lengths[best_idx]:.1f} px",
                    (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
    result['debug_image'] = debug_img
    return result


# ── compute_mm_per_pixel() ────────────────────────────────────────────────────

def compute_mm_per_pixel(
    pixel_length: float,
    known_length_mm: float,
) -> float:
    """
    Compute the physical scale factor.

    Parameters
    ----------
    pixel_length   : measured pixel span of the known scale object
    known_length_mm: physical length of the scale object in millimetres

    Returns
    -------
    mm_per_pixel : float
    """
    if pixel_length <= 0:
        raise ValueError(
            "pixel_length must be > 0. "
            "Run detect_scale_object() first."
        )
    return known_length_mm / pixel_length


print('detect_scale_object()  and  compute_mm_per_pixel()  defined.')
print('  Methods available: auto | checkerboard | canny_hough')


In [ ]:
# ── quantify_calibration_uncertainty() ───────────────────────────────────────

def quantify_calibration_uncertainty(
    pixel_lengths: list,
    known_length_mm: float,
    confidence_level: float = 0.95,
) -> dict:
    """
    Estimate 1-sigma uncertainty on mm_per_pixel from repeated measurements.

    If only one measurement is available, the function falls back to a
    conservative single-pixel rounding uncertainty.

    Parameters
    ----------
    pixel_lengths    : list of N independent pixel-length measurements
                       (e.g. from multiple edge-detections or manual picks)
    known_length_mm  : physical length of the scale object (mm)
    confidence_level : level for the confidence interval (default 0.95)

    Returns
    -------
    dict with keys
        'mm_per_pixel'             : best estimate (mean)
        'mm_per_pixel_uncertainty' : 1-sigma uncertainty
        'relative_error_pct'       : uncertainty as % of best estimate
        'n_measurements'           : number of samples used
        'ci_low', 'ci_high'        : confidence-interval bounds on mm_per_pixel
        'passes_gate'              : True if relative_error_pct < 2.0
    """
    pixel_lengths = np.asarray(pixel_lengths, dtype=float)
    n = len(pixel_lengths)

    mpp_samples = known_length_mm / pixel_lengths
    best = float(np.mean(mpp_samples))

    if n == 1:
        # Conservative half-pixel rounding uncertainty
        sigma = known_length_mm / (pixel_lengths[0] ** 2) * 0.5
        ci_lo = ci_hi = best
        print("[WARNING] Only one measurement — uncertainty is conservative (±0.5 px rounding).")
    else:
        sigma = float(np.std(mpp_samples, ddof=1))
        sem = sigma / math.sqrt(n)
        t_crit = sp_stats.t.ppf((1 + confidence_level) / 2, df=n - 1)
        ci_lo = best - t_crit * sem
        ci_hi = best + t_crit * sem

    rel_err_pct = (sigma / best) * 100.0
    passes = rel_err_pct < 2.0

    return {
        'mm_per_pixel':             best,
        'mm_per_pixel_uncertainty': sigma,
        'relative_error_pct':       rel_err_pct,
        'n_measurements':           n,
        'ci_low':                   float(ci_lo),
        'ci_high':                  float(ci_hi),
        'passes_gate':              passes,
    }


print('quantify_calibration_uncertainty()  defined.')

In [ ]:
# ── define_roi_interactive() ──────────────────────────────────────────────
# Replaced with a Colab-safe version:
#   - plt.show(block=False) so the kernel does not freeze
#   - input() wrapped with a 120-second timeout
#   - Falls back to full-frame ROI if no input is received

_roi_coords = {'x0': None, 'y0': None, 'x1': None, 'y1': None}


def define_roi_interactive(frame):
    """
    Display the reference frame and collect ROI coordinates.

    In Colab the interactive RectangleSelector cannot be used.
    Instead the image is displayed and the user types the four
    pixel coordinates when prompted.  The prompt waits up to
    ROI_INPUT_TIMEOUT_S seconds before falling back to the full frame.

    Expected input format (see Cell 24 for a worked example):
        x0,y0,x1,y1   e.g.  362,41,1085,726
        where x0,y0 = top-left corner  and  x1,y1 = bottom-right corner
        all values in pixels measured from the top-left of the full frame.
    """
    import threading

    h, w = frame.shape[:2]

    # Show the frame so the user can read pixel coordinates
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(
        'Read the pixel coordinates of your ROI from this image.\n'
        'Then type them in the input box below  (format: x0,y0,x1,y1)',
        fontsize=11
    )
    # Overlay a grid every 100 px to help read coordinates
    for xg in range(0, w, 100):
        ax.axvline(xg, color='cyan', lw=0.4, alpha=0.5)
        ax.text(xg+2, 12, str(xg), color='cyan', fontsize=6)
    for yg in range(0, h, 100):
        ax.axhline(yg, color='cyan', lw=0.4, alpha=0.5)
        ax.text(4, yg+12, str(yg), color='cyan', fontsize=6)
    ax.axis('off')
    plt.tight_layout()
    plt.show()   # non-blocking in Colab

    # Timed input — waits ROI_INPUT_TIMEOUT_S seconds
    user_input = [None]

    def _ask():
        try:
            user_input[0] = input(
                'Enter ROI as  x0,y0,x1,y1  (pixels, top-left to bottom-right)\n'
                'Example from checkerboard photo: 362,41,1085,726\n'
                'Press Enter to use the full frame as ROI: '
            ).strip()
        except Exception:
            pass

    t = threading.Thread(target=_ask, daemon=True)
    t.start()
    t.join(timeout=ROI_INPUT_TIMEOUT_S)

    raw_roi = user_input[0]

    if raw_roi:
        try:
            parts = [int(v.strip()) for v in raw_roi.replace(' ', ',').split(',') if v.strip()]
            if len(parts) == 4:
                x0, y0, x1, y1 = parts
                # Clamp to frame bounds
                x0 = max(0, min(x0, w))
                y0 = max(0, min(y0, h))
                x1 = max(0, min(x1, w))
                y1 = max(0, min(y1, h))
                roi = [x0, y0, x1, y1]
                print('ROI set from input: ' + str(roi))
            else:
                print('[WARNING] Expected 4 values — using full frame.')
                roi = [0, 0, w, h]
        except ValueError:
            print('[WARNING] Could not parse input — using full frame.')
            roi = [0, 0, w, h]
    else:
        roi = [0, 0, w, h]
        print('No input received (timeout or empty) — using full frame: ' + str(roi))

    # Reference counting lines
    global REFERENCE_LINES_PX
    ref_lines = list(REFERENCE_LINES_PX)

    if not ref_lines:
        lines_input = [None]

        def _ask_lines():
            try:
                lines_input[0] = input(
                    'Enter Y-coordinates of counting lines (pixels from top),\n'
                    'comma-separated  e.g. 300,500,700  or press Enter to skip: '
                ).strip()
            except Exception:
                pass

        t2 = threading.Thread(target=_ask_lines, daemon=True)
        t2.start()
        t2.join(timeout=ROI_INPUT_TIMEOUT_S)

        raw_lines = lines_input[0]
        if raw_lines:
            try:
                ref_lines = [int(v.strip()) for v in raw_lines.split(',') if v.strip()]
            except ValueError:
                ref_lines = []

    return {'roi_coords': roi, 'reference_lines_px': ref_lines}


print('define_roi_interactive()  defined  (Colab-safe, 2-min timeout).')


In [ ]:
# ── plot_calibration_overlay() ────────────────────────────────────────────────
# Paste this cell immediately BEFORE the cell that calls
# plot_calibration_overlay(...).  It must run once to define the function.

def plot_calibration_overlay(
    frame,
    roi_coords,
    reference_lines_px,
    mm_per_pixel,
    column_diameter_mm,
    calibration_info,
):
    """
    Draw and display a full calibration validation overlay on the reference frame.

    Shows:
      - Green rectangle  : ROI boundary (the region tracked in NB-01 through NB-05)
      - Cyan dashed lines: reference counting lines (bubble frequency measurement)
      - Yellow bar       : column diameter span at mid-ROI height
      - White scale bar  : 10 mm reference length in the bottom-left of the ROI
      - Info panel       : mm/pixel, relative error, gate status, n measurements

    Parameters
    ----------
    frame              : BGR image as returned by cv2.imread
    roi_coords         : [x0, y0, x1, y1]  ROI in full-frame pixel coordinates
    reference_lines_px : list of Y pixel positions of counting lines (full-frame)
    mm_per_pixel       : calibrated scale factor (mm per pixel)
    column_diameter_mm : physical column inner diameter (mm)
    calibration_info   : dict from quantify_calibration_uncertainty()
                         keys: mm_per_pixel, relative_error_pct, ci_low,
                               ci_high, passes_gate, n_measurements
    """
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    vis = frame.copy()
    x0, y0, x1, y1 = roi_coords
    roi_w_px = x1 - x0
    roi_h_px = y1 - y0

    # ── 1. ROI rectangle (green) ──────────────────────────────────────────────
    cv2.rectangle(vis, (x0, y0), (x1, y1), (0, 255, 0), 2)
    cv2.putText(vis, 'ROI', (x0 + 4, y0 + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # ── 2. Reference counting lines (cyan dashed) ─────────────────────────────
    for ly in reference_lines_px:
        # Draw dashed line across the ROI width
        dash_len, gap_len = 18, 10
        xx = x0
        toggle = True
        while xx < x1:
            xe = min(xx + (dash_len if toggle else gap_len), x1)
            if toggle:
                cv2.line(vis, (xx, ly), (xe, ly), (255, 255, 0), 2)
            xx = xe
            toggle = not toggle
        label = 'y=%d px' % ly
        cv2.putText(vis, label, (x1 + 4, ly + 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0), 1)

    # ── 3. Column diameter span (yellow horizontal bar at mid-ROI) ────────────
    col_diam_px = int(round(column_diameter_mm / mm_per_pixel))
    mid_y       = (y0 + y1) // 2
    cx          = (x0 + x1) // 2
    bar_x0      = cx - col_diam_px // 2
    bar_x1      = cx + col_diam_px // 2
    cv2.line(vis, (bar_x0, mid_y), (bar_x1, mid_y), (0, 215, 255), 3)
    # End tick marks
    cv2.line(vis, (bar_x0, mid_y - 8), (bar_x0, mid_y + 8), (0, 215, 255), 2)
    cv2.line(vis, (bar_x1, mid_y - 8), (bar_x1, mid_y + 8), (0, 215, 255), 2)
    cv2.putText(vis,
                'col diam = %.1f mm (%d px)' % (column_diameter_mm, col_diam_px),
                (bar_x0, mid_y - 14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 215, 255), 1)

    # ── 4. Scale bar — 10 mm reference in bottom-left of ROI ─────────────────
    scale_mm      = 10.0
    scale_px      = int(round(scale_mm / mm_per_pixel))
    sb_x0         = x0 + 12
    sb_y          = y1 - 20
    sb_x1         = sb_x0 + scale_px
    cv2.line(vis, (sb_x0, sb_y), (sb_x1, sb_y), (255, 255, 255), 3)
    cv2.line(vis, (sb_x0, sb_y - 5), (sb_x0, sb_y + 5), (255, 255, 255), 2)
    cv2.line(vis, (sb_x1, sb_y - 5), (sb_x1, sb_y + 5), (255, 255, 255), 2)
    cv2.putText(vis, '%.0f mm' % scale_mm,
                (sb_x0, sb_y - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    # ── 5. Info panel (top-right corner) ──────────────────────────────────────
    gate_ok  = calibration_info.get('passes_gate', False)
    rel_err  = calibration_info.get('relative_error_pct', 0.0)
    ci_lo    = calibration_info.get('ci_low',  mm_per_pixel)
    ci_hi    = calibration_info.get('ci_high', mm_per_pixel)
    n_meas   = calibration_info.get('n_measurements', 1)
    gate_str = 'PASS' if gate_ok else 'FAIL'
    gate_col = (0, 200, 0) if gate_ok else (0, 0, 220)

    lines_info = [
        'mm/px = %.5f' % mm_per_pixel,
        'err   = %.2f %%' % rel_err,
        'CI95  = [%.5f, %.5f]' % (ci_lo, ci_hi),
        'N meas= %d' % n_meas,
        'Gate  = %s' % gate_str,
    ]
    panel_x = x0 + 6
    panel_y = y0 + 18
    line_h  = 18
    # Background rectangle
    panel_h = len(lines_info) * line_h + 6
    panel_w = 260
    sub     = vis[panel_y - 16 : panel_y - 16 + panel_h,
                  panel_x - 4  : panel_x - 4  + panel_w]
    if sub.size > 0:
        vis[panel_y - 16 : panel_y - 16 + panel_h,
            panel_x - 4  : panel_x - 4  + panel_w] = (sub * 0.45).astype('uint8')

    for k, txt in enumerate(lines_info):
        col = gate_col if 'Gate' in txt else (255, 255, 255)
        cv2.putText(vis, txt,
                    (panel_x, panel_y + k * line_h),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.48, col, 1)

    # ── 6. Display with matplotlib ────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 9))
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(
        'Calibration overlay — ROI (green)  |  '
        'counting lines (cyan)  |  column diameter (yellow)  |  '
        'scale bar (white)',
        fontsize=10
    )
    ax.axis('off')

    # Legend patches
    legend = [
        mpatches.Patch(color='lime',    label='ROI boundary'),
        mpatches.Patch(color='yellow',  label='Counting lines'),
        mpatches.Patch(color='#00D7FF', label='Column diameter'),
        mpatches.Patch(color='white',   label='10 mm scale bar'),
    ]
    ax.legend(handles=legend, loc='lower right',
              fontsize=8, framealpha=0.75)

    plt.tight_layout()
    plt.show()
    print('Overlay displayed.')
    print('  ROI           : %s  (%d x %d px)' % (roi_coords, roi_w_px, roi_h_px))
    print('  ROI physical  : %.1f x %.1f mm' % (roi_w_px * mm_per_pixel, roi_h_px * mm_per_pixel))
    print('  Scale bar     : %.0f mm = %d px' % (scale_mm, scale_px))
    print('  Ref lines     : %s' % reference_lines_px)
    return vis


print('plot_calibration_overlay()  defined.')

In [ ]:
# ── save_config() ─────────────────────────────────────────────────────────────

def save_config(
    mm_per_pixel: float,
    mm_per_pixel_uncertainty: float,
    fps: float,
    roi_coords: list,
    column_diameter_mm: float,
    reference_lines_px: list,
    schema_version: str,
    output_path: Path,
    extra_meta: dict | None = None,
) -> Path:
    """
    Write config.json conforming to Appendix A, Section A.1 of the roadmap.

    Parameters
    ----------
    mm_per_pixel              : calibrated scale factor (mm/px)
    mm_per_pixel_uncertainty  : 1-sigma uncertainty on the scale factor
    fps                       : validated camera frame rate
    roi_coords                : [x0, y0, x1, y1] bounding box (pixels)
    column_diameter_mm        : physical column inner diameter (mm)
    reference_lines_px        : list of Y-coordinates of counting lines (px)
    schema_version            : schema version tag
    output_path               : directory where config.json will be written
    extra_meta                : optional dict of additional metadata fields

    Returns
    -------
    Path to the written config.json file
    """
    config = {
        # ── Mandatory fields (Appendix A.1) ──────────────────────────────────
        "mm_per_pixel":              round(mm_per_pixel, 8),
        "mm_per_pixel_uncertainty":  round(mm_per_pixel_uncertainty, 8),
        "fps":                       float(fps),
        "roi_coords":                [int(v) for v in roi_coords],
        "column_diameter_mm":        float(column_diameter_mm),
        "reference_lines_px":        [int(v) for v in reference_lines_px],
        "schema_version":            schema_version,
        "created_at":                datetime.now(timezone.utc).isoformat(),
    }

    # Optional metadata extensions
    if extra_meta:
        config.update(extra_meta)

    out_file = Path(output_path) / "config.json"
    with open(out_file, 'w') as f:
        json.dump(config, f, indent=2)

    print(f"config.json written → {out_file.resolve()}")
    return out_file


# ── load_config() — used by ALL downstream notebooks ─────────────────────────

def load_config(path: str = "config.json") -> dict:
    """
    Load and validate config.json.

    Raises
    ------
    FileNotFoundError  if the file does not exist
    KeyError           if any mandatory field is missing
    """
    MANDATORY_FIELDS = [
        'mm_per_pixel', 'mm_per_pixel_uncertainty', 'fps',
        'roi_coords', 'column_diameter_mm', 'reference_lines_px',
        'schema_version', 'created_at',
    ]
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"config.json not found at '{p.resolve()}'.\n"
            "Run NB-00 (calibration) first — no downstream notebook may "
            "execute without a validated config.json."
        )
    with open(p) as f:
        cfg = json.load(f)

    missing = [k for k in MANDATORY_FIELDS if k not in cfg]
    if missing:
        raise KeyError(
            f"config.json is missing mandatory fields: {missing}.\n"
            "Re-run NB-00 to produce a schema-compliant config.json."
        )
    return cfg


print('save_config()  and  load_config()  defined.')

---
## Cell 3 — Load Reference Frame and Run Scale Detection

Tries `'checkerboard'` first (two-pass, sub-pixel, 89 measurements).
Falls back to `'canny_hough'` automatically if the board is not in frame.

In [ ]:
# ── LOAD REFERENCE FRAME ──────────────────────────────────────────────────────
# Loads frame_bgr for use in Cell 3a (wall diagnostic) and Cell 3b (manual
# calibration). Checkerboard detection is skipped — use the manual
# column-width path in Cells 3a and 3b instead.

ref_path = Path(REFERENCE_FRAME_PATH)
if not ref_path.exists():
    raise FileNotFoundError(
        'Reference frame not found: ' + str(ref_path.resolve()) + '\n'
        'Set REFERENCE_FRAME_PATH in Cell 1 to the full absolute path.'
    )

frame_bgr = cv2.imread(str(ref_path))
if frame_bgr is None:
    raise IOError(
        'cv2 cannot read ' + str(ref_path) + '\n'
        'File may be corrupted. Re-run NB-00-pre to extract a fresh frame.'
    )

h_px, w_px = frame_bgr.shape[:2]
print('Reference frame loaded.')
print(f'  Path  : {ref_path.resolve()}')
print(f'  Size  : {w_px} x {h_px} px')
print(f'  Format: {ref_path.suffix.upper()}')
print()
print('Proceed to Cell 3a to read wall positions.')

# Display the frame
fig, ax = plt.subplots(figsize=(4, 12))
ax.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
ax.set_title('Reference frame', fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.show()

# scale_result is built in Cell 3b from manual wall measurements.
# Initialise to None so downstream cells can detect if Cell 3b was skipped.
scale_result = None

In [ ]:
# ── COLUMN WIDTH DIAGNOSTIC ───────────────────────────────────────────────────
# Displays a horizontal slice at mid-height with pixel grid.
# Read the left and right wall x-positions from the intensity profile,
# then set LEFT_WALL_PX and RIGHT_WALL_PX in Cell 1.

_frame = frame_bgr.copy()
_h, _w = _frame.shape[:2]
_mid   = _h // 2
_slice = cv2.cvtColor(_frame[_mid-100:_mid+100, :], cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(14, 3))
ax.imshow(_slice)
for x in range(0, _w, 20):
    ax.axvline(x, color='cyan', lw=0.3, alpha=0.6)
    if x % 100 == 0:
        ax.text(x+2, 5, str(x), color='cyan', fontsize=7, va='top')
ax.set_title('Mid-column slice — read left and right wall x-pixel positions')
ax.set_xlabel('x pixel position')
ax.set_yticks([])
plt.tight_layout()
plt.show()

_gray = cv2.cvtColor(_frame[_mid:_mid+1, :], cv2.COLOR_BGR2GRAY).flatten().astype(float)
fig2, ax2 = plt.subplots(figsize=(14, 2.5))
ax2.plot(_gray, color='orange', lw=1.0)
ax2.set_xlabel('x pixel position')
ax2.set_ylabel('Intensity')
ax2.set_title('Intensity profile at mid-height — walls show as bright edges')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Frame size: {_w} x {_h} px')
print('Set LEFT_WALL_PX and RIGHT_WALL_PX in Cell 1, then run Cell 3b.')

In [ ]:
# ── APPLY MANUAL COLUMN-WIDTH CALIBRATION ─────────────────────────────────────
if LEFT_WALL_PX >= 0 and RIGHT_WALL_PX > 0 and RIGHT_WALL_PX > LEFT_WALL_PX:
    _col_width_px = RIGHT_WALL_PX - LEFT_WALL_PX
    _mpp          = COLUMN_WIDTH_CALIB_MM / _col_width_px

    scale_result = {
        'method':            'manual_column_width',
        'pixel_length':      float(_col_width_px),
        'pixel_lengths_all': [float(_col_width_px)],
        'lines':             [float(_col_width_px)],
        'board_corners':     None,
    }

    print('Manual column-width calibration applied.')
    print(f'  Left wall      : {LEFT_WALL_PX} px')
    print(f'  Right wall     : {RIGHT_WALL_PX} px')
    print(f'  Column width   : {_col_width_px} px')
    print(f'  Physical width : {COLUMN_WIDTH_CALIB_MM} mm')
    print(f'  mm_per_pixel   : {_mpp:.6f}')
    print()
    print('scale_result ready. Proceed to Cell 4.')

elif LEFT_WALL_PX == 0 and RIGHT_WALL_PX == 0:
    raise RuntimeError(
        'LEFT_WALL_PX and RIGHT_WALL_PX are both 0.\n'
        'Set them in Cell 1 from the Cell 3a diagnostic, then re-run.'
    )
else:
    raise ValueError(
        f'Invalid wall positions: LEFT={LEFT_WALL_PX}  RIGHT={RIGHT_WALL_PX}\n'
        'RIGHT_WALL_PX must be greater than LEFT_WALL_PX and both must be > 0.'
    )

---
### Cell 3c — Debug: Confirm scale_result
Verify Cell 3 ran correctly before proceeding.

In [ ]:
try:
    print("scale_result exists.")
    print("Detection method:", scale_result.get("method"))
    print("Pixel length:", scale_result.get("pixel_length"))
    print("Number of measurements:", len(scale_result.get("pixel_lengths_all", [])))
except NameError:
    print("scale_result does not exist yet. Run Cell 3 first.")

---
## Cell 4 — Compute mm/pixel and Quantify Uncertainty

When the checkerboard is detected, `pixel_lengths_all` contains all 89 corner
spacings — the Student-t CI path is activated automatically.
When Hough is used, falls back to the conservative half-pixel estimate.

In [ ]:
# Use all individual inter-corner spacings for a rigorous uncertainty estimate.
# Checkerboard:  scale_result['lines'] = list of 89 spacings (px)
# Canny/Hough:   scale_result['lines'] = list of Hough segment lengths
#                → only one meaningful measurement; falls back to
#                  conservative half-pixel uncertainty automatically.

if scale_result['method'] == 'checkerboard':
    # All corner spacings — rich statistical sample
    pixel_length_measurements = scale_result['pixel_lengths_all']
    print(f"Using {len(pixel_length_measurements)} checkerboard corner spacings.")
else:
    # Single Hough-line length — single-element list
    pixel_length_measurements = [scale_result['pixel_length']]
    print("Using single Hough-line measurement (conservative uncertainty).")

print(f"Known physical length : {KNOWN_PHYSICAL_LENGTH_MM} mm")

# For checkerboard: known_length_mm per spacing = SQUARE_SIZE_MM (one square)
# because each spacing measures one inter-corner distance = one square side.
# For Hough: known_length_mm = KNOWN_PHYSICAL_LENGTH_MM (full reference).
if scale_result['method'] == 'checkerboard':
    uncertainty_known_mm = SQUARE_SIZE_MM   # each spacing = one square = 40 mm
else:
    uncertainty_known_mm = KNOWN_PHYSICAL_LENGTH_MM

calib = quantify_calibration_uncertainty(
    pixel_lengths=pixel_length_measurements,
    known_length_mm=uncertainty_known_mm,
)

print()
print('─' * 55)
print('CALIBRATION RESULTS')
print('─' * 55)
print(f"  Method                  : {scale_result['method']}")
print(f"  N measurements          : {calib['n_measurements']}")
print(f"  mm_per_pixel            : {calib['mm_per_pixel']:.6f} mm/px")
print(f"  1-sigma uncertainty     : {calib['mm_per_pixel_uncertainty']:.6f} mm/px")
print(f"  Relative error          : {calib['relative_error_pct']:.4f} %")
print(f"  95 % CI                 : [{calib['ci_low']:.6f}, {calib['ci_high']:.6f}]")
gate_sym = '✅  PASSES' if calib['passes_gate'] else '❌  FAILS'
print(f"  Gate (< 2 % rel. error) : {gate_sym}")
print('─' * 55)

if not calib['passes_gate']:
    print(
        "\n[GATE FAILED] Calibration error ≥ 2 %.\n"
        "Actions:\n"
        "  1. Verify CHECKERBOARD_GRID matches the actual corner count.\n"
        "  2. Ensure the board is fully visible and in focus.\n"
        "  3. Try CALIBRATION_MODE='checkerboard' to see the explicit error.\n"
        "  4. If using canny_hough: verify KNOWN_PHYSICAL_LENGTH_MM is correct.\n"
        "Downstream notebooks CANNOT run until this gate passes."
    )
else:
    print("\nCalibration gate passed — proceed to ROI definition.")


---
## Cell 5 — FPS Validation (optional)
Reads the video file header to confirm the frame rate matches `VIDEO_FPS`.
Leave `REFERENCE_VIDEO_PATH = ''` to skip and trust the configured value.

In [ ]:
# ── FPS VALIDATION ──────────────────────────────────────────────────────
# The YAML provides VIDEO_FPS, but this cell checks the actual video header when
# the video is available. The YAML value remains the fallback.

validated_fps = VIDEO_FPS

video_path_for_fps = Path(REFERENCE_VIDEO_PATH) if REFERENCE_VIDEO_PATH else None
if video_path_for_fps and video_path_for_fps.exists():
    cap = cv2.VideoCapture(str(video_path_for_fps))
    header_fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    if header_fps and header_fps > 0:
        validated_fps = float(header_fps)
        rel_diff = abs(validated_fps - VIDEO_FPS) / max(VIDEO_FPS, 1e-9) * 100
        print('FPS read from video header:', round(validated_fps, 6))
        print('FPS from YAML             :', round(VIDEO_FPS, 6))
        print('Relative difference (%)   :', round(rel_diff, 4))
        if rel_diff > 1.0:
            print('WARNING: video FPS differs from YAML by more than 1%.')
    else:
        print('Could not read FPS from video header. Using YAML VIDEO_FPS:', VIDEO_FPS)
else:
    print('Reference video not found or not configured. Using YAML VIDEO_FPS:', VIDEO_FPS)


---
## Cell 6 — ROI Definition

**Prerequisite: Cell 3 must have run successfully** (it creates `scale_result` with the board corner coordinates).

Three paths in order of priority:
- **Path A** — `ROI_COORDS_OVERRIDE` set in Cell 1 → used immediately, no prompt
- **Path B** — checkerboard corners detected in Cell 3 → auto-set from board
- **Path C** — interactive prompt (120s timeout) → type coordinates manually

In [ ]:
# ── ROI DEFINITION — SAFE OVERRIDE ONLY ──────────────────────────────────────
# ROI and reference lines are always taken from Cell 1 variables.
# Interactive prompt and board-corner path are disabled — they produced
# wrong values when the checkerboard was absent.
#
# To change the ROI: edit ROI_COORDS_OVERRIDE in Cell 1 and re-run from Cell 1.

# ── Prerequisite check ────────────────────────────────────────────────────────
if scale_result is None:
    raise RuntimeError(
        'scale_result is None.\n'
        'Run Cell 3b (manual calibration) before running this cell.'
    )

# ── Validate ROI_COORDS_OVERRIDE is set ───────────────────────────────────────
if not ROI_COORDS_OVERRIDE or len(ROI_COORDS_OVERRIDE) != 4:
    raise ValueError(
        'ROI_COORDS_OVERRIDE is not set or has wrong length.\n'
        'Set ROI_COORDS_OVERRIDE = [x0, y0, x1, y1] in Cell 1.\n'
        'Current value: ' + str(ROI_COORDS_OVERRIDE)
    )

if not REF_LINES_OVERRIDE or len(REF_LINES_OVERRIDE) != 3:
    raise ValueError(
        'REF_LINES_OVERRIDE is not set or does not have exactly 3 values.\n'
        'Set REF_LINES_OVERRIDE = [y1, y2, y3] in Cell 1 (ROI-local Y coordinates).\n'
        'Current value: ' + str(REF_LINES_OVERRIDE)
    )

# ── Assign from override ──────────────────────────────────────────────────────
roi_coords      = [int(v) for v in ROI_COORDS_OVERRIDE]
reference_lines = [int(v) for v in REF_LINES_OVERRIDE]

x0, y0, x1, y1 = roi_coords
roi_w = x1 - x0
roi_h = y1 - y0

# ── Sanity checks ─────────────────────────────────────────────────────────────
if roi_w <= 0 or roi_h <= 0:
    raise ValueError(
        f'Invalid ROI: width={roi_w} px  height={roi_h} px.\n'
        'x1 must be > x0 and y1 must be > y0.'
    )

h_frame, w_frame = frame_bgr.shape[:2]
if x0 < 0 or y0 < 0 or x1 > w_frame or y1 > h_frame:
    raise ValueError(
        f'ROI [{x0},{y0},{x1},{y1}] extends outside the frame '
        f'({w_frame} x {h_frame} px).\n'
        'Adjust ROI_COORDS_OVERRIDE in Cell 1.'
    )

for rl in reference_lines:
    if not (0 <= rl <= roi_h):
        raise ValueError(
            f'Reference line y={rl} is outside ROI height (0–{roi_h} px).\n'
            'REF_LINES_OVERRIDE values must be ROI-local Y coordinates.\n'
            'Adjust REF_LINES_OVERRIDE in Cell 1.'
        )

# ── Report ────────────────────────────────────────────────────────────────────
_col_w_px = RIGHT_WALL_PX - LEFT_WALL_PX if RIGHT_WALL_PX > LEFT_WALL_PX else 1
_roi_mm   = roi_w * COLUMN_WIDTH_CALIB_MM / _col_w_px
print('ROI set from Cell 1 override.')
print(f'  roi_coords      : {roi_coords}')
print(f'  Width           : {roi_w} px  ({_roi_mm:.1f} mm)')
print(f'  Height          : {roi_h} px')
print(f'  reference_lines : {reference_lines}  (ROI-local Y px)')
print()
print('Safe ROI assignment complete — proceed to Cell 7 (overlay).')

---
## Cell 7 — Overlay Validation: ROI + Reference Lines + mm-per-pixel

The overlay is also saved as a PNG to Drive for permanent audit trail.

In [ ]:
overlay_bgr = plot_calibration_overlay(
    frame       = frame_bgr,
    roi_coords  = roi_coords,
    reference_lines_px = reference_lines,
    mm_per_pixel       = calib['mm_per_pixel'],
    column_diameter_mm = COLUMN_DIAMETER_MM,
    calibration_info   = calib,
)

CALIBRATION_OVERLAY_PATH = Path(
    '/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A/calibration_overlay.png'
)
CALIBRATION_OVERLAY_PATH.parent.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(CALIBRATION_OVERLAY_PATH), overlay_bgr)
print('Overlay PNG written →', CALIBRATION_OVERLAY_PATH.resolve())


---
## Cell 8 — Acceptance Gate Check

In [ ]:
# ── Formal gate verification before writing config.json ───────────────────────

print("═" * 55)
print("STAGE 0  ·  ACCEPTANCE GATE")
print("═" * 55)
print(f"  mm_per_pixel             : {calib['mm_per_pixel']:.6f} mm/px")
print(f"  Uncertainty (1σ)         : {calib['mm_per_pixel_uncertainty']:.6f} mm/px")
print(f"  Relative calibration err : {calib['relative_error_pct']:.4f} %")
print(f"  Gate threshold           : {MAX_RELATIVE_CALIBRATION_ERROR_PCT:.3f} %")

gate_pass = calib['passes_gate']
if gate_pass:
    print("  Gate result              : ✅  PASS")
    print()
    print("  config.json will be written.")
    print("  Downstream notebooks are authorised to run.")
else:
    print("  Gate result              : ❌  FAIL")
    print()
    print("  [BLOCKED] config.json will NOT be written.")
    print("  Fix calibration error before proceeding.")
    print("  Check the YAML calibration settings, reference frame quality, or collect more samples.")

print("═" * 55)

---
## Cell 9 — Write config.json

In [ ]:
_SETUP = Path('/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A')
CONFIG_JSON_PATH = _SETUP / 'config.json'
if not gate_pass:
    raise RuntimeError(
        "Calibration gate FAILED — config.json not written.\n"
        "Resolve the calibration error and re-run from Cell 3."
    )

extra = {
    # Provenance fields
    'project_root':          str(DRIVE_ROOT),
    'setup_id':             'setup_A',
    'run_label':             RUN_LABEL if 'RUN_LABEL' in dir() else 'run_001',
    'source_video':         str(REFERENCE_VIDEO_PATH),
    'reference_frame':      str(Path(REFERENCE_FRAME_PATH).name),
    'reference_frame_path': str(Path(REFERENCE_FRAME_PATH)),
    'video_resolution_px':  [VIDEO_WIDTH_PX, VIDEO_HEIGHT_PX],
    'calibration_method':   scale_result['method'],
    'n_measurements':       calib['n_measurements'],
    'relative_error_pct':   round(calib['relative_error_pct'], 6),
    'ci_95_low':            round(calib['ci_low'], 8),
    'ci_95_high':           round(calib['ci_high'], 8),
    'known_reference_mm':   KNOWN_PHYSICAL_LENGTH_MM,
    'gate_threshold_pct':   MAX_RELATIVE_CALIBRATION_ERROR_PCT,
    'calibration_overlay':  str(CALIBRATION_OVERLAY_PATH),
    # Checkerboard-specific provenance
    'checkerboard_grid':    list(CHECKERBOARD_GRID) if scale_result['method'] == 'checkerboard' else None,
    'square_size_mm':       SQUARE_SIZE_MM if scale_result['method'] == 'checkerboard' else None,
    'board_corners_px':     scale_result.get('board_corners'),
    'column_width_mm':          500.0,
    'column_height_mm':         750.0,
    'column_depth_mm':          1200.0,
    'column_cross_section_mm2': 250000.0,
    'hydraulic_diameter_mm':    500.0,
    'sparger_y_top_px':         3400,
    'sparger_y_bot_px':         3600,
    'water_surface_y_px':       450,
    'active_height_mm':         682.9,
}

config_path = save_config(
    mm_per_pixel             = calib['mm_per_pixel'],
    mm_per_pixel_uncertainty = calib['mm_per_pixel_uncertainty'],
    fps                      = validated_fps,
    roi_coords               = roi_coords,
    column_diameter_mm       = COLUMN_DIAMETER_MM,
    reference_lines_px       = reference_lines,
    schema_version           = SCHEMA_VERSION,
    output_path              = OUTPUT_PATH,
    extra_meta               = extra,
)


with open(config_path) as f:
    print(json.dumps(json.load(f), indent=2))


In [ ]:
print('LEFT_WALL_PX :', LEFT_WALL_PX)
print('RIGHT_WALL_PX:', RIGHT_WALL_PX)
print('Width px     :', RIGHT_WALL_PX - LEFT_WALL_PX)
print('mm_per_pixel :', COLUMN_WIDTH_CALIB_MM / (RIGHT_WALL_PX - LEFT_WALL_PX))

In [ ]:
# ── SAFE SAVE VERIFICATION ────────────────────────────────────────────────────
import json as _json

_cfg_written = Path(OUTPUT_PATH) / 'config.json'
if not _cfg_written.exists():
    raise RuntimeError(
        f'config.json was NOT written to {_cfg_written.resolve()}\n'
        'Check OUTPUT_PATH in Cell 1 and Drive permissions.'
    )

_cfg_check = _json.load(open(_cfg_written))
_checks = {
    'mm_per_pixel': (0.20 <= _cfg_check['mm_per_pixel'] <= 0.25,
                           f"{_cfg_check['mm_per_pixel']:.6f}"),
    'roi_coords':         (_cfg_check['roi_coords'] == [0, 450, 2160, 3400],
                           str(_cfg_check['roi_coords'])),
    'column_diameter_mm': (_cfg_check['column_diameter_mm'] == 500.0,
                           str(_cfg_check['column_diameter_mm'])),
    'fps':                (_cfg_check['fps'] > 40.0,
                           str(round(_cfg_check['fps'], 4))),
}

print('=' * 55)
print('config.json SAFE SAVE VERIFICATION')
print('=' * 55)
all_ok = True
for field, (ok, val) in _checks.items():
    status = 'OK  ' if ok else 'FAIL'
    print(f'  {status}  {field:<25} : {val}')
    if not ok:
        all_ok = False
print('=' * 55)

if not all_ok:
    raise RuntimeError(
        'config.json was written but contains unexpected values.\n'
        'Review the FAIL fields above and re-run from Cell 1.'
    )

print('config.json verified and ready for NB-01.')
print(f'Path: {_cfg_written.resolve()}')

---
## Cell 10 — Round-Trip Validation

In [ ]:
# ── Read back the file and verify all mandatory fields are present ─────────────

cfg_verify = load_config(str(config_path))

# Verify round-trip fidelity of mm_per_pixel
rt_error = abs(cfg_verify['mm_per_pixel'] - calib['mm_per_pixel'])
assert rt_error < 1e-6, f"Round-trip mm_per_pixel mismatch: {rt_error}"

# Verify ROI coordinates
assert cfg_verify['roi_coords'] == [int(v) for v in roi_coords], \
    "Round-trip ROI mismatch"

print("Round-trip validation  : ✅  PASSED")
print(f"Schema version         : {cfg_verify['schema_version']}")
print(f"Created at             : {cfg_verify['created_at']}")
print()
print("═" * 55)
print("NB-00  ·  CALIBRATION COMPLETE")
print("═" * 55)
print(f"  config.json  → {config_path.resolve()}")
print(f"  Overlay PNG  → {CALIBRATION_OVERLAY_PATH.resolve()}")
print()
print("Gate:  ✅  calibration error within YAML threshold  —  NB-01 is authorised to run.")
print("═" * 55)

---
## Cell 11 — Summary Report
Copy this output into your lab notebook or gate documentation.

In [ ]:
from IPython.display import Markdown, display

report_md = f"""
## NB-00 Calibration Gate Report

| Field | Value |
|---|---|
| **Schema version** | {cfg_verify['schema_version']} |
| **Created at** | {cfg_verify['created_at']} |
| **Reference frame** | {cfg_verify.get('reference_frame', REFERENCE_FRAME_PATH)} |
| **Known physical reference (mm)** | {KNOWN_PHYSICAL_LENGTH_MM} |
| **Calibration method** | {scale_result['method']} |
| **N measurements** | {calib['n_measurements']} |
| **mm per pixel** | {calib['mm_per_pixel']:.6f} |
| **Uncertainty 1σ (mm/px)** | {calib['mm_per_pixel_uncertainty']:.6f} |
| **Relative error (%)** | {calib['relative_error_pct']:.4f} |
| **95 % CI** | [{calib['ci_low']:.6f}, {calib['ci_high']:.6f}] |
| **Gate threshold from YAML** | {MAX_RELATIVE_CALIBRATION_ERROR_PCT:.3f}% → {'✅ PASS' if gate_pass else '❌ FAIL'} |
| **Validated FPS** | {validated_fps:.4f} |
| **ROI coords (px)** | {roi_coords} |
| **Column diameter (mm)** | {COLUMN_DIAMETER_MM} |
| **Reference lines (px)** | {reference_lines} |

**Decision:** {'NB-01 (preprocessing) is authorised to run.' if gate_pass else 'Gate failed — calibration must be re-done.'}
"""
display(Markdown(report_md))